In [268]:
import pandas as pd
import numpy as np
from pathlib import Path

# Setup

In [269]:
# Data path
data_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data')

data_files_dict = {'index_1030' : ['index_1030 Gender Development Index.xlsx', '2025_metadata_Gender Development Index 2025.xlsx'],
                   'index_9' : ['index_9 EGDI.xlsx', '20251102_metadata_E-Government Development Index_2024.xlsx'],
                   'index_1032': ['20260710_historical data_index_1032_SDG 2026.xlsx', '20260710_metadata_Sustainable_Development_Index_2026.xlsx']}

index_name = 'index_9'
index_data_files = data_files_dict[index_name]

index_data_path = Path(data_path) / 'Historical' / index_data_files[0]
index_metadata_path = Path(data_path) / 'Metadata' / index_data_files[1]

# Read raw data
index_data_raw = pd.read_excel(index_data_path)
index_metadata_raw = pd.read_excel(index_metadata_path)
index_metadata_raw['Arab Countries'] = index_metadata_raw['Arab Countries'].str.strip()

country_iso = pd.read_excel(Path(data_path) / 'metadata_SDG_2026.xlsx', sheet_name='country')
country_iso['country'] = country_iso['country'].str.strip()
country_iso['ISO'] = country_iso['ISO'].str.strip()
# - data correction
if index_name == 'index_1032':
    index_metadata_raw['Index_ID'] = '1032'
    index_metadata_raw = index_metadata_raw.merge(country_iso, how='left', left_on='Arab Countries', right_on='country')
    # make first row columns names
    index_data_raw.columns = index_data_raw.iloc[0]
    #index_data_raw = index_data_raw[1:]

In [270]:
arab_countries = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                  'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']
# Index data minor reformatting and filtering
index_data = index_data_raw.copy()
index_data = index_data.rename(columns={'Unnamed: 0':'KeyID','Unnamed: 1':'Series name','Unnamed: 2':'Year',
                                        'year':'Year'})
index_data = index_data.iloc[1:]

# Metadata
index_metadata = index_metadata_raw.copy()
index_metadata = index_metadata.rename(columns={'year of historical data': 'year historical',
                                                'year historical ':'year historical',
                                                'Year for each country, year of the simulator ':'Year for each country',
                                                'data simulator':'data'})

# Data consistency
index_metadata['year historical'] = (pd.to_numeric(index_metadata['year historical'], errors='coerce').astype('Int64'))
index_metadata['Year for each country'] = (pd.to_numeric(index_metadata['Year for each country'], errors='coerce').astype('Int64'))
index_metadata['year of the report'] = (pd.to_numeric(index_metadata['year of the report'], errors='coerce').astype('Int64'))
index_metadata['KPI ID'] = index_metadata['KPI ID'].astype(str)
index_metadata['KPI ID'] = index_metadata['KPI ID'].str.replace('.0','')

# Most recent historical data

In [271]:
# Composite key
newest_index_data = index_data.copy()

newest_index_data = newest_index_data.melt(id_vars=['KeyID','Year'], 
                                           var_name='Country', value_name='data_historical_automated')        # convert wide to long
newest_index_data = newest_index_data.dropna(subset=['data_historical_automated'])                            # drop na
newest_index_data = newest_index_data[newest_index_data['Country'].isin(arab_countries)] # filter for arab countries
# groupby and extract most recent year value
max_year_idx = newest_index_data.groupby(['KeyID', 'Country'])['Year'].idxmax()
newest_index_data = newest_index_data.loc[max_year_idx].reset_index(drop=True)

# Reformat values
newest_index_data['data_historical_automated'] = newest_index_data['data_historical_automated'].astype('Float64')

# Rename composite key (KeyID, Country) ==> (KPI_ID, ISO) to match metadata format
newest_index_data = newest_index_data.rename(columns={'KeyID':'KPI ID', 'Country':'ISO', 'Year':'year_historical_automated'})
newest_index_data['KPI ID'] = newest_index_data['KPI ID'].astype(int).astype(str)

In [272]:
# Composite key
data_long_format = index_data.copy()

data_long_format = data_long_format.melt(id_vars=['KeyID','Year'], 
                                           var_name='Country', value_name='data_historical_automated')        # convert wide to long
data_long_format = data_long_format.dropna(subset=['data_historical_automated'])                            # drop na
data_long_format = data_long_format[data_long_format['Country'].isin(arab_countries)]

In [273]:
index_data

,KeyID,Series name,Year,AFG,ALB,DZA,AND,AGO,ATG,ARG,...,GBR,USA,URY,UZB,VUT,VEN,VNM,YEM,ZMB,ZWE
1,282,Mean years of schooling,2022,3,11.3,8.1,10.6,5.4,9.3,11.1,...,13.4,13.7,9,11.9,7.1,11.1,8.4,3.2,7.2,8.7
2,282,Mean years of schooling,1990,1.5,7.8,3.6,NaN,NaN,NaN,7.9,...,7.9,12.3,7.2,NaN,NaN,4.6,3.9,0.3,4.7,4.5
3,282,Mean years of schooling,1991,1.6,7.8,3.8,NaN,NaN,NaN,8,...,8.6,12.4,7.2,NaN,NaN,4.8,4.1,0.4,5,4.7
4,282,Mean years of schooling,1992,1.6,7.8,4,NaN,NaN,NaN,8,...,9.3,12.5,7.2,NaN,NaN,5,4.2,0.4,5.2,4.9
5,282,Mean years of schooling,1993,1.7,7.8,4.3,NaN,NaN,NaN,8.1,...,10,12.6,7.3,NaN,NaN,5.1,4.3,0.5,5.5,5.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,274,Active mobile-broadband subscriptions per 100 ...,2020,19,69.6,90.2,88.5,19.8,52.3,68.7,...,109,154,108,93.4,354,43.8,79.6,4.56,53.6,56
293,274,Active mobile-broadband subscriptions per 100 ...,2021,18.6,72.2,95.9,87.4,21.2,52,72.8,...,112,164,110,105,297,48.7,86.5,19.5,52.8,59.1
294,274,Active mobile-broadband subscriptions per 100 ...,2022,56.3,75.7,98.4,95.4,26.2,51.7,76.2,...,114,172,117,106,339,52.3,95.5,25.7,54.9,60.5
295,274,Active mobile-broadband subscriptions per 100 ...,2023,55.5,74.1,104,107,30.3,NaN,NaN,...,115,185,110,107,NaN,NaN,99.8,34.3,60.2,67.6


In [274]:
# Discrepancies between Index and historical data
index_metadata_kpis = index_metadata['KPI ID'].unique().tolist()
historical_index_kpis = newest_index_data['KPI ID'].unique().tolist()
result = list(set(index_metadata_kpis) - set(historical_index_kpis))
result

[]

# Recreate metadata

In [275]:
# Use current file as template
metadata_automated = index_metadata[['Index_ID','KPI name in the report ','Arab Countries','ISO','KPI ID','Year for each country', 'year of the report', 
                                     'data', 'year historical', 'data historical', 'Data availability and treatment',
                                    'Year difference', 'last year available used to compute the index', 'Value difference',
                                    'Accuracy problem']].copy()

# join in automated data pull
metadata_automated = metadata_automated.merge(newest_index_data, how='left', on=['KPI ID', 'ISO'])
# Recreate calculated fields
# G 'Year for each country'
# I 'year of the report'
#metadata_check = index_metadata_check.merge(metadata_recreate[['KPI ID', 'ISO', 'Year', 'data']], how='left', on=['KPI ID','ISO'])

### QA Metrics

In [276]:
data_hist_col = 'data_historical_automated'
year_hist_col = 'year_historical_automated'

metadata_automated['value_difference'] = abs(metadata_automated['data']-metadata_automated[data_hist_col])
metadata_automated['value_difference_percentage'] = np.where(
    (metadata_automated['value_difference'] == 0) & (metadata_automated['data'] == 0),
    0,
    metadata_automated['value_difference'] / metadata_automated['data']
)

metadata_automated['value_equal'] = (
    (metadata_automated['value_difference_percentage'] < 0.01).fillna(False)
)


In [277]:
# Match from previous years

# Value does not match most previous years data
mismatch = metadata_automated[~metadata_automated['value_equal']]

mismatch = mismatch[['ISO', 'KPI ID', 'data']].copy()

# Join in previous data
index_data_long = index_data.copy()
index_data_long = index_data_long.melt(id_vars=['KeyID','Year'], 
                                           var_name='Country', value_name='data_historical_previous_year')        # convert wide to long
index_data_long = index_data_long.dropna(subset=['data_historical_previous_year'])                            # drop na
index_data_long = index_data_long[index_data_long['Country'].isin(arab_countries)]
index_data_long = index_data_long.rename(columns={'KeyID':'KPI ID', 'Country':'ISO', 'Year':'year_historical_previous'})
index_data_long['KPI ID'] = index_data_long['KPI ID'].astype(int).astype(str)
index_data_long['year_historical_previous'] = index_data_long['year_historical_previous'].astype(int)

# Merge
mismatch = mismatch.merge(index_data_long, how='left', on=['KPI ID', 'ISO'])
mismatch = mismatch[mismatch['year_historical_previous'] >=2019]
mismatch['value_difference_previous_year'] = abs(mismatch['data']- mismatch['data_historical_previous_year'])
mismatch['value_difference_pct_previous_year'] = mismatch['value_difference_previous_year']/mismatch['data']
mismatch = mismatch[mismatch['value_difference_pct_previous_year']<0.0075] # within 0.75%
# Select based on most recent year
YEAR = 'year_historical_previous'
TOL = 1e-9 # tolerance
mismatch['is_zero'] = mismatch['value_difference_pct_previous_year'].abs() < TOL

ranked = mismatch.sort_values(
    ['ISO', 'KPI ID', 'is_zero', YEAR],
    ascending=[True, True, False, False]
)
ranked['selection_rank'] = ranked.groupby(['ISO', 'KPI ID']).cumcount() + 1
ranked['match_previous_year'] = ranked['selection_rank'] == 1
ranked = ranked.reset_index(drop=True)

previous_year_match = ranked.copy()
previous_year_match = previous_year_match[previous_year_match['match_previous_year']]
previous_year_match = previous_year_match[['ISO', 'KPI ID','year_historical_previous', 'data_historical_previous_year', 'value_difference_previous_year', 'value_difference_pct_previous_year',
                                          'match_previous_year']]

print(metadata_automated.shape)
metadata_automated = metadata_automated.merge(previous_year_match, how='left', on=['KPI ID', 'ISO'])
metadata_automated['year_difference'] = metadata_automated['year_historical_automated']-metadata_automated['year_historical_previous']
metadata_automated['match_previous_year'] =metadata_automated['match_previous_year'].fillna(False)

print(metadata_automated.shape)

(315, 20)
(315, 26)


In [278]:
metadata_automated['Record_Quality_Good'] = np.where(metadata_automated['value_equal'],
                                                    1,
                                                    0)
metadata_automated['Record_Quality_Old'] = np.where(metadata_automated['match_previous_year'],
                                                    1,
                                                    0)
metadata_automated['Record_Quality_No_Historical_Data_Available']= np.where(metadata_automated['data_historical_automated'].isna(),
                                                    1,
                                                    0)

metadata_automated['Record_Quality_Inaccurate_Imputed'] = np.where( ((metadata_automated['Record_Quality_Good']==0) & (metadata_automated['Record_Quality_Old']==0)) & (metadata_automated['Record_Quality_No_Historical_Data_Available']==0),
                                                                   1,
                                                                   0)
metadata_automated['Record_Quality_Total'] = metadata_automated['Record_Quality_Total'] = metadata_automated['Record_Quality_Good']+metadata_automated['Record_Quality_Old']+metadata_automated['Record_Quality_Inaccurate_Imputed']


### Summary Stats

In [279]:
total_quality_good = sum(metadata_automated['Record_Quality_Good'])
total_quality_old = sum(metadata_automated['Record_Quality_Old'])
total_quality_inaccurate_imputed = sum(metadata_automated['Record_Quality_Inaccurate_Imputed'])
total_quality_no_data_available = sum(metadata_automated['Record_Quality_No_Historical_Data_Available'])

total_records = total_quality_good+total_quality_old+total_quality_inaccurate_imputed+total_quality_no_data_available


if metadata_automated.shape[0] == total_records:
    print(f"All {metadata_automated.shape[0]} records accounted for.")
else:
    print(f"Record count mismatch {total_records} of {metadata_automated.shape[0]}")

summary_stats = pd.DataFrame({
    'Category': ['Good', 'Old', 'Inaccurate_Imputed', 'No_Historical_Data_Available'],
    'Count': [total_quality_good, total_quality_old, total_quality_inaccurate_imputed, total_quality_no_data_available],
})
summary_stats['Percentage'] = (summary_stats['Count'] / total_records * 100).round(2)
# Add a totals row
summary_stats.loc['Total'] = ['Total', total_records, 100.0]

All 315 records accounted for.


### Summary Stats KPI

In [280]:
summary_stats_kpi = metadata_automated.groupby(['KPI ID'])[
    ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate_Imputed', 'Record_Quality_No_Historical_Data_Available']
].sum().reset_index()

summary_stats_kpi['Total'] = (
    summary_stats_kpi['Record_Quality_Good']
    + summary_stats_kpi['Record_Quality_Old']
    + summary_stats_kpi['Record_Quality_Inaccurate_Imputed']
    + summary_stats_kpi['Record_Quality_No_Historical_Data_Available']
)
# Add percentage columns
for col in ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate_Imputed', 'Record_Quality_No_Historical_Data_Available']:
    summary_stats_kpi[col + '_pct'] = (summary_stats_kpi[col] / summary_stats_kpi['Total']).round(2)

### Summary Stats Country

In [281]:
summary_stats_country = metadata_automated.groupby(['Arab Countries'])[
    ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate_Imputed', 'Record_Quality_No_Historical_Data_Available']
].sum().reset_index()

summary_stats_country['Total'] = (
    summary_stats_country['Record_Quality_Good']
    + summary_stats_country['Record_Quality_Old']
    + summary_stats_country['Record_Quality_Inaccurate_Imputed']
    + summary_stats_country['Record_Quality_No_Historical_Data_Available']
)

# Add percentage columns
for col in ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate_Imputed', 'Record_Quality_No_Historical_Data_Available']:
    summary_stats_country[col + '_pct'] = (summary_stats_country[col] / summary_stats_country['Total']).round(2)

In [282]:

from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
def write_dataframes_to_excel(
    dataframes: dict,
    output_path: str = "output.xlsx",
    freeze_header: bool = True,
    autofit_columns: bool = True,
    max_col_width: int = 50,
    header_bg_color: str = None,#"4472C4",   # Excel-blue; set None for no fill
    header_font_color: str = "FFFFFF",
    header_bold: bool = True,
    add_borders: bool = False,
    add_autofilter: bool = True,
    index: bool = False,
):
    """
    Write a dict of DataFrames to one Excel file, one sheet per DataFrame.
 
    Parameters
    ----------
    dataframes : dict
        Mapping of {sheet_name: dataframe}. Sheet names are what you control
        naming for — just use whatever keys you want.
        Example: {"Sales": df_sales, "Inventory": df_inv}
    output_path : str
        Path to save the .xlsx file.
    freeze_header : bool
        If True, freezes the header row so it stays visible when scrolling.
    autofit_columns : bool
        If True, resizes each column to fit its longest value.
    max_col_width : int
        Cap on auto-fit column width (prevents huge columns from long text).
    header_bg_color : str or None
        Hex color (no '#') for header cell background fill.
    header_font_color : str
        Hex color (no '#') for header font color.
    header_bold : bool
        Whether header text is bold.
    add_borders : bool
        If True, adds thin borders around all cells.
    add_autofilter : bool
        If True, adds filter dropdowns to the header row.
    index : bool
        Whether to write the DataFrame index as a column.
    """
 
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dataframes.items():
            # Excel sheet names max 31 chars, no : \ / ? * [ ]
            safe_name = _sanitize_sheet_name(sheet_name)
            df.to_excel(writer, sheet_name=safe_name, index=index)
 
            worksheet = writer.sheets[safe_name]
            _format_worksheet(
                worksheet,
                df,
                freeze_header=freeze_header,
                autofit_columns=autofit_columns,
                max_col_width=max_col_width,
                header_bg_color=header_bg_color,
                header_font_color=header_font_color,
                header_bold=header_bold,
                add_borders=add_borders,
                add_autofilter=add_autofilter,
                index=index,
            )
 
    print(f"Saved: {output_path}")
 
 
def _sanitize_sheet_name(name: str) -> str:
    invalid_chars = [":", "\\", "/", "?", "*", "[", "]"]
    for ch in invalid_chars:
        name = name.replace(ch, "")
    return name[:31]
 
 
def _format_worksheet(
    ws,
    df,
    freeze_header,
    autofit_columns,
    max_col_width,
    header_bg_color,
    header_font_color,
    header_bold,
    add_borders,
    add_autofilter,
    index,
):
    n_cols = df.shape[1] + (1 if index else 0)
    n_rows = df.shape[0] + 1  # +1 for header row
 
    # --- Header styling ---
    header_font = Font(bold=header_bold, color=header_font_color if header_bg_color else "000000")
    header_fill = PatternFill(
        start_color=header_bg_color, end_color=header_bg_color, fill_type="solid"
    ) if header_bg_color else None
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
 
    for col_idx in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        if header_fill:
            cell.fill = header_fill
        cell.alignment = header_align
 
    # Slightly taller header row to fit wrapped text
    ws.row_dimensions[1].height = 22
 
    # --- Optional thin borders on all used cells ---
    if add_borders:
        thin = Side(style="thin", color="B7B7B7")
        border = Border(left=thin, right=thin, top=thin, bottom=thin)
        for row in ws.iter_rows(min_row=1, max_row=n_rows, min_col=1, max_col=n_cols):
            for cell in row:
                cell.border = border
 
    # --- Freeze header row (and index column if present) ---
    if freeze_header:
        ws.freeze_panes = "B2" if index else "A2"
 
    # --- Autofilter on header row ---
    if add_autofilter:
        ws.auto_filter.ref = ws.dimensions
 
    # --- Autofit column widths based on longest value ---
    if autofit_columns:
        for col_idx in range(1, n_cols + 1):
            col_letter = get_column_letter(col_idx)
            max_len = 0
            for cell in ws[col_letter]:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[col_letter].width = min(max_len + 3, max_col_width)

In [283]:

file_name = f"Metadata_Automation_{index_name}_V2.xlsx"
output_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data') / file_name

sheets = {
    'Metadata_automated' : metadata_automated,
    'Summary Stats' : summary_stats,
    'Summary Stats by KPI' : summary_stats_kpi,
    'Summary Stats by Country' : summary_stats_country,
    'Raw Data Long format' : data_long_format
}

write_dataframes_to_excel(
        sheets,
        output_path=output_path,
        freeze_header=True,
        autofit_columns=True,
        #header_bg_color="4472C4",
        #header_font_color="FFFFFF",
        header_bold=True,
        add_borders=True,
        add_autofilter=True,
        index=False,
    )

Saved: C:\Users\User\OneDrive\Desktop\UN\Data\Metadata_Automation_index_9_V2.xlsx
